In [ ]:
import requests
from bs4 import BeautifulSoup
import re

# URL of the syllabus page
syllabus_url = 'https://www.icsbook.info/member-syllabus'

try:
    print(f"Scraping links from {syllabus_url}...")
    response = requests.get(syllabus_url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')

    # Find all links containing 'read-book'
    links = soup.find_all('a', href=re.compile(r'read-book/\d+'))

    # Extract unique IDs and full URLs
    all_book_links = sorted(list(set([link['href'] for link in links])))
    new_target_ids = sorted(list(set(re.findall(r'/(\d+)', " ".join(all_book_links)))), key=int)

    print(f"Successfully found {len(new_target_ids)} unique book IDs.")

    # Update target_ids for subsequent cells
    target_ids = new_target_ids

    # Display first few for confirmation
    print("Sample links found:")
    for link in all_book_links[:5]:
        print(link)

except Exception as e:
    print(f"An error occurred while scraping: {e}")

Scraping links from https://www.icsbook.info/member-syllabus...
Successfully found 211 unique book IDs.
Sample links found:
/read-book/100
/read-book/101
/read-book/102
/read-book/103
/read-book/104


In [ ]:
import requests
import json
import os
import shutil
from google.colab import files

json_dir = 'book_jsons'
os.makedirs(json_dir, exist_ok=True)

# We use target_ids populated by the scraper in the previous cell
print(f"Processing {len(target_ids)} books from syllabus...")

index_data = []

for book_id in target_ids:
    file_path = os.path.join(json_dir, f"{book_id}.json")

    # Check if we already have it to save bandwidth
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        try:
            res = requests.get(f"https://www.icsbook.info/details-json/{book_id}", timeout=25)
            if res.status_code == 200:
                data = res.json()
                with open(file_path, 'w', encoding='utf-8') as f:
                    json.dump(data, f)
            else:
                continue
        except:
            continue

    # Collect metadata for index
    index_data.append({
        "id": str(book_id),
        "title": data.get('book_title', f"Book {book_id}"),
        "has_desc": bool(data.get('book_description') and data.get('book_description').strip())
    })

# Save updated index
with open('book_index.json', 'w', encoding='utf-8') as f:
    json.dump(index_data, f)

# Zip the updated collection
shutil.make_archive('all_book_jsons', 'zip', json_dir)

print(f"Finished! Metadata for {len(index_data)} books processed.")
print("Downloading updated files...")

files.download('all_book_jsons.zip')
files.download('book_index.json')
files.download('optimized_viewer.html') # Downloading again to ensure it is in the same folder as the new JSONs

Processing 211 books from syllabus...
Finished! Metadata for 211 books processed.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
import json
import os
import re

# Configuration
base_url = 'https://www.icsbook.info/details-json/'
output_dir = 'book_descriptions'
os.makedirs(output_dir, exist_ok=True)

# Get the list of URLs from the notebook cell (simulated via regex on the provided list)
book_list_text = """https://www.icsbook.info/read-book/626
https://www.icsbook.info/read-book/632
https://www.icsbook.info/read-book/617
https://www.icsbook.info/read-book/623
https://www.icsbook.info/read-book/621
https://www.icsbook.info/read-book/633
https://www.icsbook.info/read-book/712
https://www.icsbook.info/read-book/628
https://www.icsbook.info/read-book/618
https://www.icsbook.info/read-book/620
https://www.icsbook.info/read-book/713
https://www.icsbook.info/read-book/634
https://www.icsbook.info/read-book/624
https://www.icsbook.info/read-book/631
https://www.icsbook.info/read-book/619
https://www.icsbook.info/read-book/629
https://www.icsbook.info/read-book/630
https://www.icsbook.info/read-book/627
https://www.icsbook.info/read-book/714
https://www.icsbook.info/read-book/715
https://www.icsbook.info/read-book/716
https://www.icsbook.info/read-book/717
https://www.icsbook.info/read-book/84
https://www.icsbook.info/read-book/100
https://www.icsbook.info/read-book/101
https://www.icsbook.info/read-book/102
https://www.icsbook.info/read-book/103
https://www.icsbook.info/read-book/104
https://www.icsbook.info/read-book/105
https://www.icsbook.info/read-book/106
https://www.icsbook.info/read-book/107
https://www.icsbook.info/read-book/108
https://www.icsbook.info/read-book/109
https://www.icsbook.info/read-book/110
https://www.icsbook.info/read-book/111
https://www.icsbook.info/read-book/112
https://www.icsbook.info/read-book/113
https://www.icsbook.info/read-book/114
https://www.icsbook.info/read-book/115
https://www.icsbook.info/read-book/116
https://www.icsbook.info/read-book/117
https://www.icsbook.info/read-book/118
https://www.icsbook.info/read-book/119
https://www.icsbook.info/read-book/120
https://www.icsbook.info/read-book/121
https://www.icsbook.info/read-book/132
https://www.icsbook.info/read-book/79
https://www.icsbook.info/read-book/63
https://www.icsbook.info/read-book/90
https://www.icsbook.info/read-book/91
https://www.icsbook.info/read-book/92
https://www.icsbook.info/read-book/93
https://www.icsbook.info/read-book/94
https://www.icsbook.info/read-book/95
https://www.icsbook.info/read-book/96
https://www.icsbook.info/read-book/97
https://www.icsbook.info/read-book/98
https://www.icsbook.info/read-book/99
https://www.icsbook.info/read-book/77
https://www.icsbook.info/read-book/80
https://www.icsbook.info/read-book/718
https://www.icsbook.info/read-book/720
https://www.icsbook.info/read-book/149
https://www.icsbook.info/read-book/193
https://www.icsbook.info/read-book/719
https://www.icsbook.info/read-book/201
https://www.icsbook.info/read-book/202
https://www.icsbook.info/read-book/203
https://www.icsbook.info/read-book/204
https://www.icsbook.info/read-book/198
https://www.icsbook.info/read-book/153
https://www.icsbook.info/read-book/155
https://www.icsbook.info/read-book/145
https://www.icsbook.info/read-book/146
https://www.icsbook.info/read-book/307
https://www.icsbook.info/read-book/308
https://www.icsbook.info/read-book/309
https://www.icsbook.info/read-book/30
https://www.icsbook.info/read-book/318
https://www.icsbook.info/read-book/323
https://www.icsbook.info/read-book/320
https://www.icsbook.info/read-book/324
https://www.icsbook.info/read-book/744
https://www.icsbook.info/read-book/721
https://www.icsbook.info/read-book/530
https://www.icsbook.info/read-book/321
https://www.icsbook.info/read-book/331
https://www.icsbook.info/read-book/338
https://www.icsbook.info/read-book/342
https://www.icsbook.info/read-book/291
https://www.icsbook.info/read-book/292
https://www.icsbook.info/read-book/293
https://www.icsbook.info/read-book/333
https://www.icsbook.info/read-book/334
https://www.icsbook.info/read-book/335
https://www.icsbook.info/read-book/337
https://www.icsbook.info/read-book/343
https://www.icsbook.info/read-book/329
https://www.icsbook.info/read-book/314
https://www.icsbook.info/read-book/315
https://www.icsbook.info/read-book/316
https://www.icsbook.info/read-book/317
https://www.icsbook.info/read-book/707
https://www.icsbook.info/read-book/351
https://www.icsbook.info/read-book/352
https://www.icsbook.info/read-book/350
https://www.icsbook.info/read-book/326
https://www.icsbook.info/read-book/327
https://www.icsbook.info/read-book/472
https://www.icsbook.info/read-book/332
https://www.icsbook.info/read-book/507
https://www.icsbook.info/read-book/508
https://www.icsbook.info/read-book/509
https://www.icsbook.info/read-book/510
https://www.icsbook.info/read-book/511
https://www.icsbook.info/read-book/512
https://www.icsbook.info/read-book/513
https://www.icsbook.info/read-book/514
https://www.icsbook.info/read-book/515
https://www.icsbook.info/read-book/363
https://www.icsbook.info/read-book/362
https://www.icsbook.info/read-book/645
https://www.icsbook.info/read-book/722
https://www.icsbook.info/read-book/723
https://www.icsbook.info/read-book/724
https://www.icsbook.info/read-book/361
https://www.icsbook.info/read-book/456
https://www.icsbook.info/read-book/356
https://www.icsbook.info/read-book/286
https://www.icsbook.info/read-book/699
https://www.icsbook.info/read-book/365
https://www.icsbook.info/read-book/28
https://www.icsbook.info/read-book/27
https://www.icsbook.info/read-book/366
https://www.icsbook.info/read-book/675
https://www.icsbook.info/read-book/367
https://www.icsbook.info/read-book/369
https://www.icsbook.info/read-book/700
https://www.icsbook.info/read-book/370
https://www.icsbook.info/read-book/368
https://www.icsbook.info/read-book/372
https://www.icsbook.info/read-book/32
https://www.icsbook.info/read-book/373
https://www.icsbook.info/read-book/322
https://www.icsbook.info/read-book/371
https://www.icsbook.info/read-book/741
https://www.icsbook.info/read-book/374
https://www.icsbook.info/read-book/375
https://www.icsbook.info/read-book/376
https://www.icsbook.info/read-book/380
https://www.icsbook.info/read-book/381
https://www.icsbook.info/read-book/382
https://www.icsbook.info/read-book/326
https://www.icsbook.info/read-book/644
https://www.icsbook.info/read-book/377
https://www.icsbook.info/read-book/726
https://www.icsbook.info/read-book/386
https://www.icsbook.info/read-book/725
https://www.icsbook.info/read-book/387
https://www.icsbook.info/read-book/385
https://www.icsbook.info/read-book/384
https://www.icsbook.info/read-book/701
https://www.icsbook.info/read-book/383
https://www.icsbook.info/read-book/389
https://www.icsbook.info/read-book/390
https://www.icsbook.info/read-book/642
https://www.icsbook.info/read-book/393
https://www.icsbook.info/read-book/439
https://www.icsbook.info/read-book/391
https://www.icsbook.info/read-book/742
https://www.icsbook.info/read-book/740
https://www.icsbook.info/read-book/388
https://www.icsbook.info/read-book/271
https://www.icsbook.info/read-book/396
https://www.icsbook.info/read-book/394
https://www.icsbook.info/read-book/395
https://www.icsbook.info/read-book/519
https://www.icsbook.info/read-book/253
https://www.icsbook.info/read-book/280
https://www.icsbook.info/read-book/398
https://www.icsbook.info/read-book/400
https://www.icsbook.info/read-book/281
https://www.icsbook.info/read-book/399
https://www.icsbook.info/read-book/727
https://www.icsbook.info/read-book/641
https://www.icsbook.info/read-book/733
https://www.icsbook.info/read-book/734
https://www.icsbook.info/read-book/735
https://www.icsbook.info/read-book/736
https://www.icsbook.info/read-book/584
https://www.icsbook.info/read-book/305
https://www.icsbook.info/read-book/302
https://www.icsbook.info/read-book/448
https://www.icsbook.info/read-book/739
https://www.icsbook.info/read-book/304
https://www.icsbook.info/read-book/306
https://www.icsbook.info/read-book/730
https://www.icsbook.info/read-book/310
https://www.icsbook.info/read-book/262
https://www.icsbook.info/read-book/747
https://www.icsbook.info/read-book/729
https://www.icsbook.info/read-book/737
https://www.icsbook.info/read-book/311
https://www.icsbook.info/read-book/312
https://www.icsbook.info/read-book/732
https://www.icsbook.info/read-book/301
https://www.icsbook.info/read-book/731
https://www.icsbook.info/read-book/687
https://www.icsbook.info/read-book/686
https://www.icsbook.info/read-book/278
https://www.icsbook.info/read-book/646
https://www.icsbook.info/read-book/279"""

# Extract IDs from URLs
target_ids = re.findall(r'/(\d+)', book_list_text)

print(f"Targeting {len(target_ids)} specific books...")

found_count = 0
for book_id in target_ids:
    try:
        response = requests.get(f"{base_url}{book_id}", timeout=5)
        if response.status_code == 200:
            data = response.json()
            description = data.get('book_description')

            if description and description.strip():
                file_path = os.path.join(output_dir, f"book_{book_id}.txt")
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(description)
                found_count += 1

        if int(book_id) % 20 == 0:
            print(f"Processed through ID {book_id}...")
    except:
        continue

print(f"\nFinished! Saved {found_count} descriptions to /{output_dir}")

Targeting 212 specific books...
Processed through ID 620...
Processed through ID 100...
Processed through ID 120...
Processed through ID 80...
Processed through ID 720...
Processed through ID 320...
Processed through ID 700...
Processed through ID 380...
Processed through ID 740...
Processed through ID 280...
Processed through ID 400...

Finished! Saved 78 descriptions to /book_descriptions


In [ ]:
import shutil
from google.colab import files

# Zip the directory containing the descriptions
shutil.make_archive('book_descriptions', 'zip', 'book_descriptions')

# Trigger the download
files.download('book_descriptions.zip')

print("Zipping complete. The download should start automatically.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Zipping complete. The download should start automatically.


In [ ]:
import os

# Identify which books were actually found by looking at the saved files
files_found = os.listdir('book_descriptions')
successful_ids = [f.replace('book_', '').replace('.txt', '') for f in files_found if f.endswith('.txt')]

print(f"Found {len(successful_ids)} book descriptions. Here are the links:")
for b_id in sorted(successful_ids, key=int):
    print(f"https://www.icsbook.info/read-book/{b_id}")

Found 77 book descriptions. Here are the links:
https://www.icsbook.info/read-book/27
https://www.icsbook.info/read-book/28
https://www.icsbook.info/read-book/30
https://www.icsbook.info/read-book/32
https://www.icsbook.info/read-book/63
https://www.icsbook.info/read-book/77
https://www.icsbook.info/read-book/80
https://www.icsbook.info/read-book/132
https://www.icsbook.info/read-book/153
https://www.icsbook.info/read-book/155
https://www.icsbook.info/read-book/193
https://www.icsbook.info/read-book/253
https://www.icsbook.info/read-book/262
https://www.icsbook.info/read-book/271
https://www.icsbook.info/read-book/280
https://www.icsbook.info/read-book/281
https://www.icsbook.info/read-book/291
https://www.icsbook.info/read-book/292
https://www.icsbook.info/read-book/293
https://www.icsbook.info/read-book/304
https://www.icsbook.info/read-book/305
https://www.icsbook.info/read-book/306
https://www.icsbook.info/read-book/307
https://www.icsbook.info/read-book/308
https://www.icsbook.inf

In [ ]:
import requests
import json

# Sample one book to see the full JSON structure
sample_id = '63'
response = requests.get(f'https://www.icsbook.info/details-json/{sample_id}')
if response.status_code == 200:
    sample_data = response.json()
    print("JSON Keys available:", sample_data.keys())
    print("\nSample Data (excluding full description):")
    # Display everything except the long description for a quick look
    display({k: v for k, v in sample_data.items() if k != 'book_description'})
else:
    print("Could not fetch sample JSON.")

JSON Keys available: dict_keys(['id', 'book_title', 'book_writer', 'book_translator', 'book_section', 'book_image', 'book_pdf', 'book_audio', 'book_audio_hq', 'book_syllabus', 'book_category', 'book_description', 'total_view', 'total_share', 'total_download', 'download_date', 'slug', 'status', 'created_at', 'updated_at'])

Sample Data (excluding full description):


{'id': 63,
 'book_title': 'আল কুরআনের শৈল্পিক সৌন্দর্য',
 'book_writer': '459',
 'book_translator': 'মুহাম্মদ খলিলুর রহমান মুমিন',
 'book_section': None,
 'book_image': 'uploads/bookbook_image/Slide1.jpg',
 'book_pdf': 'http://shibircloud.com/pdf/al_quraner_shoilpik_soundorjo.pdf',
 'book_audio': None,
 'book_audio_hq': None,
 'book_syllabus': None,
 'book_category': None,
 'total_view': 6934,
 'total_share': 117,
 'total_download': 0,
 'download_date': '2021-03-30 17:56:00',
 'slug': '15168',
 'status': 1,
 'created_at': '2020-01-14 15:42:32',
 'updated_at': '2026-08-21 17:25:01'}

In [ ]:
import os
import requests
import json

def create_master_html(output_filename='master_books.html'):
    source_dir = 'book_descriptions'
    if not os.path.exists(source_dir):
        print(f"Error: '{source_dir}' directory not found. Please ensure the cells that download the books have been executed in this session.")
        return

    files_found = sorted([f for f in os.listdir(source_dir) if f.endswith('.txt')],
                         key=lambda x: int(x.replace('book_', '').replace('.txt', '')))

    html_content = ["""<html>
<head>
    <meta charset='UTF-8'>
    <title>Master Index of Book Descriptions</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, sans-serif; line-height: 1.6; max-width: 1000px; margin: auto; padding: 40px; background-color: #f4f7f6; color: #333; }
        .container { background: white; padding: 30px; border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }
        .book-entry { border-bottom: 2px solid #eee; margin-bottom: 50px; padding-bottom: 30px; }
        .toc { background: #eef2f3; padding: 25px; border-radius: 8px; margin-bottom: 40px; }
        h1 { color: #2c3e50; text-align: center; }
        .meta { color: #555; font-style: italic; margin-bottom: 15px; border-left: 4px solid #3498db; padding-left: 10px; font-size: 0.9em; }
        .description-content { background: #fff; padding: 20px; border: 1px solid #ddd; border-radius: 5px; }
        a { color: #3498db; text-decoration: none; }
        a:hover { text-decoration: underline; }
    </style>
</head>
<body id='top'>
    <div class='container'>
    <h1>Master Book Index</h1>
    <div class='toc'>
        <h3>Table of Contents</h3>
        <ul>"""]

    book_details = []

    print(f"Processing {len(files_found)} books and fetching metadata...")
    for filename in files_found:
        book_id = filename.replace('book_', '').replace('.txt', '')
        desc_path = os.path.join(source_dir, filename)

        with open(desc_path, 'r', encoding='utf-8') as f:
            description_html = f.read()

        try:
            res = requests.get(f'https://www.icsbook.info/details-json/{book_id}', timeout=5)
            data = res.json() if res.status_code == 200 else {}
        except:
            data = {}

        title = data.get('book_title', f"Book {book_id}")
        writer = data.get('book_writer', 'N/A')
        translator = data.get('book_translator', 'N/A')

        book_details.append({
            'id': book_id,
            'title': title,
            'writer': writer,
            'translator': translator,
            'content': description_html
        })

        html_content.append(f'            <li><a href="#book_{book_id}">{title}</a></li>')

    html_content.append("        </ul>\n    </div>\n    <hr>")

    # Generate individual book sections
    for book in book_details:
        html_content.append(f"""
    <div id='book_{book['id']}' class='book-entry'>
        <h2>{book['title']}</h2>
        <div class='meta'>
            ID: {book['id']} | Writer ID: {book['writer']} | Translator: {book['translator']}
        </div>
        <div class='description-content'>
            <body>
                {book['content']}
            </body>
        </div>
        <p style='text-align: right;'><a href='#top'>↑ Back to Index</a></p>
    </div>""")

    html_content.append("    </div>\n</body>\n</html>")

    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("\n".join(html_content))

    print(f"\nSuccess! Created master file: '{output_filename}'")

create_master_html()

Processing 77 books and fetching metadata...

Success! Created master file: 'master_books.html'


In [ ]:
from google.colab import files

# Trigger the download of the master index file
if os.path.exists('master_books.html'):
    files.download('master_books.html')
    print("Download of 'master_books.html' initiated.")
else:
    print("Error: master_books.html not found.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download of 'master_books.html' initiated.


In [ ]:
import json
import os

def generate_optimized_bundle():
    source_dir = 'book_descriptions'
    all_books = {}

    # 1. Create a compact JSON database
    files = sorted([f for f in os.listdir(source_dir) if f.endswith('.txt')], key=lambda x: int(x.replace('book_', '').replace('.txt', '')))

    print(f"Compiling {len(files)} books into JSON database...")
    for filename in files:
        book_id = filename.replace('book_', '').replace('.txt', '')
        with open(os.path.join(source_dir, filename), 'r', encoding='utf-8') as f:
            all_books[book_id] = {
                "content": f.read()
            }

    with open('books_db.json', 'w', encoding='utf-8') as f:
        json.dump(all_books, f)

    # 2. Create a lightweight HTML Viewer
    html_viewer = """<!DOCTYPE html>
<html>
<head>
    <title>Book Index Viewer</title>
    <meta charset="UTF-8">
    <style>
        body { font-family: sans-serif; display: flex; height: 100vh; margin: 0; }
        #sidebar { width: 300px; border-right: 1px solid #ccc; overflow-y: auto; background: #f8f9fa; padding: 15px; }
        #content { flex: 1; padding: 40px; overflow-y: auto; line-height: 1.6; }
        .toc-item { cursor: pointer; color: #007bff; padding: 5px 0; border-bottom: 1px solid #eee; }
        .toc-item:hover { text-decoration: underline; }
        h2 { margin-top: 0; color: #333; }
    </style>
</head>
<body>
    <div id="sidebar"><h3>Books Index</h3><div id="toc">Loading index...</div></div>
    <div id="content"><div id="viewer">Select a book from the left to read description.</div></div>

    <script>
        let bookData = {};

        fetch('books_db.json')
            .then(res => res.json())
            .then(data => {
                bookData = data;
                const toc = document.getElementById('toc');
                toc.innerHTML = '';
                Object.keys(data).forEach(id => {
                    const div = document.createElement('div');
                    div.className = 'toc-item';
                    div.innerText = 'Book ID: ' + id;
                    div.onclick = () => {
                        document.getElementById('viewer').innerHTML = '<h2>Book ' + id + '</h2>' + data[id].content;
                    };
                    toc.appendChild(div);
                });
            });
    </script>
</body>
</html>"""

    with open('viewer.html', 'w', encoding='utf-8') as f:
        f.write(html_viewer)

    print("Done! generated 'books_db.json' and 'viewer.html'.")
    print("Keep both files in the same folder to use.")

generate_optimized_bundle()

Compiling 77 books into JSON database...
Done! generated 'books_db.json' and 'viewer.html'.
Keep both files in the same folder to use.


In [ ]:
from google.colab import files
import time

# Download both files
for f in ['books_db.json', 'viewer.html']:
    files.download(f)
    time.sleep(1) # Small delay to ensure browser handles multiple downloads

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
import json
import os
import shutil
from google.colab import files

json_dir = 'book_jsons'
os.makedirs(json_dir, exist_ok=True)

print(f"Downloading {len(target_ids)} full JSON files...")

index_data = []

for book_id in target_ids:
    try:
        res = requests.get(f"https://www.icsbook.info/details-json/{book_id}", timeout=5)
        if res.status_code == 200:
            data = res.json()
            # Save the full individual JSON
            with open(os.path.join(json_dir, f"{book_id}.json"), 'w', encoding='utf-8') as f:
                json.dump(data, f)

            # Collect metadata for a tiny index file
            index_data.append({
                "id": book_id,
                "title": data.get('book_title', f"Book {book_id}"),
                "has_desc": bool(data.get('book_description'))
            })
    except:
        continue

# Save the tiny index
with open('book_index.json', 'w', encoding='utf-8') as f:
    json.dump(index_data, f)

# Zip the individual JSONs
shutil.make_archive('all_book_jsons', 'zip', json_dir)
files.download('all_book_jsons.zip')

print(f"\nFinished! Individual JSONs zipped. Created 'book_index.json' for the viewer.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Finished! Individual JSONs zipped. Created 'book_index.json' for the viewer.


In [ ]:
viewer_code = """<!DOCTYPE html>
<html>
<head>
    <title>Dynamic Book Viewer</title>
    <meta charset="UTF-8">
    <style>
        body { font-family: sans-serif; display: flex; height: 100vh; margin: 0; background: #f0f2f5; }
        #sidebar { width: 350px; border-right: 1px solid #ccc; overflow-y: auto; background: white; padding: 20px; }
        #content { flex: 1; padding: 40px; overflow-y: auto; background: #fff; }
        .book-item { cursor: pointer; padding: 10px; border-bottom: 1px solid #eee; transition: 0.2s; }
        .book-item:hover { background: #f0f7ff; }
        .no-desc { color: #999; font-style: italic; }
        .has-desc { color: #2c3e50; font-weight: bold; }
        .badge { font-size: 0.7em; padding: 2px 6px; border-radius: 4px; margin-left: 5px; }
        .badge-green { background: #d4edda; color: #155724; }
        .badge-gray { background: #e2e3e5; color: #383d41; }
        #viewer-area { max-width: 800px; margin: auto; }
    </style>
</head>
<body>
    <div id="sidebar">
        <h2>Book Index</h2>
        <input type="text" id="search" placeholder="Search titles..." style="width:100%; padding:8px; margin-bottom:15px;">
        <div id="list">Loading index...</div>
    </div>
    <div id="content">
        <div id="viewer-area">Select a book to load details dynamically.</div>
    </div>

    <script>
        let fullIndex = [];
        const listEl = document.getElementById('list');
        const viewEl = document.getElementById('viewer-area');

        // 1. Load the tiny index
        fetch('book_index.json')
            .then(res => res.json())
            .then(data => {
                fullIndex = data;
                renderList(data);
            });

        function renderList(items) {
            listEl.innerHTML = '';
            items.forEach(book => {
                const div = document.createElement('div');
                div.className = 'book-item ' + (book.has_desc ? 'has-desc' : 'no-desc');
                div.innerHTML = book.title +
                    `<span class="badge ${book.has_desc ? 'badge-green' : 'badge-gray'}">${book.has_desc ? 'Has Desc' : 'No Desc'}</span>`;

                div.onclick = () => loadBook(book.id);
                listEl.appendChild(div);
            });
        }

        // 2. Load individual JSON only when clicked
        function loadBook(id) {
            viewEl.innerHTML = 'Loading book data...';
            fetch(`book_jsons/${id}.json`)
                .then(res => res.json())
                .then(data => {
                    viewEl.innerHTML = `
                        <h1>${data.book_title}</h1>
                        <p><strong>Writer:</strong> ${data.book_writer} | <strong>Translator:</strong> ${data.book_translator || 'N/A'}</p>
                        <hr>
                        <div>${data.book_description || '<i style="color:red">No description available for this book.</i>'}</div>
                    `;
                });
        }

        document.getElementById('search').oninput = (e) => {
            const term = e.target.value.toLowerCase();
            renderList(fullIndex.filter(b => b.title.toLowerCase().includes(term)));
        };
    </script>
</body>
</html>"""

with open('optimized_viewer.html', 'w', encoding='utf-8') as f:
    f.write(viewer_code)

files.download('optimized_viewer.html')
files.download('book_index.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Full list of book links

https://www.icsbook.info/read-book/626
https://www.icsbook.info/read-book/632
https://www.icsbook.info/read-book/617
https://www.icsbook.info/read-book/623
https://www.icsbook.info/read-book/621
https://www.icsbook.info/read-book/633
https://www.icsbook.info/read-book/712
https://www.icsbook.info/read-book/628
https://www.icsbook.info/read-book/618
https://www.icsbook.info/read-book/620
https://www.icsbook.info/read-book/713
https://www.icsbook.info/read-book/634
https://www.icsbook.info/read-book/624
https://www.icsbook.info/read-book/631
https://www.icsbook.info/read-book/619
https://www.icsbook.info/read-book/629
https://www.icsbook.info/read-book/630
https://www.icsbook.info/read-book/627
https://www.icsbook.info/read-book/714
https://www.icsbook.info/read-book/715
https://www.icsbook.info/read-book/716
https://www.icsbook.info/read-book/717
https://www.icsbook.info/read-book/84
https://www.icsbook.info/read-book/100
https://www.icsbook.info/read-book/101
https://www.icsbook.info/read-book/102
https://www.icsbook.info/read-book/103
https://www.icsbook.info/read-book/104
https://www.icsbook.info/read-book/105
https://www.icsbook.info/read-book/106
https://www.icsbook.info/read-book/107
https://www.icsbook.info/read-book/108
https://www.icsbook.info/read-book/109
https://www.icsbook.info/read-book/110
https://www.icsbook.info/read-book/111
https://www.icsbook.info/read-book/112
https://www.icsbook.info/read-book/113
https://www.icsbook.info/read-book/114
https://www.icsbook.info/read-book/115
https://www.icsbook.info/read-book/116
https://www.icsbook.info/read-book/117
https://www.icsbook.info/read-book/118
https://www.icsbook.info/read-book/119
https://www.icsbook.info/read-book/120
https://www.icsbook.info/read-book/121
https://www.icsbook.info/read-book/132
https://www.icsbook.info/read-book/79
https://www.icsbook.info/read-book/63
https://www.icsbook.info/read-book/90
https://www.icsbook.info/read-book/91
https://www.icsbook.info/read-book/92
https://www.icsbook.info/read-book/93
https://www.icsbook.info/read-book/94
https://www.icsbook.info/read-book/95
https://www.icsbook.info/read-book/96
https://www.icsbook.info/read-book/97
https://www.icsbook.info/read-book/98
https://www.icsbook.info/read-book/99
https://www.icsbook.info/read-book/77
https://www.icsbook.info/read-book/80
https://www.icsbook.info/read-book/718
https://www.icsbook.info/read-book/720
https://www.icsbook.info/read-book/149
https://www.icsbook.info/read-book/193
https://www.icsbook.info/read-book/719
https://www.icsbook.info/read-book/201
https://www.icsbook.info/read-book/202
https://www.icsbook.info/read-book/203
https://www.icsbook.info/read-book/204
https://www.icsbook.info/read-book/198
https://www.icsbook.info/read-book/153
https://www.icsbook.info/read-book/155
https://www.icsbook.info/read-book/145
https://www.icsbook.info/read-book/146
https://www.icsbook.info/read-book/307
https://www.icsbook.info/read-book/308
https://www.icsbook.info/read-book/309
https://www.icsbook.info/read-book/30
https://www.icsbook.info/read-book/318
https://www.icsbook.info/read-book/323
https://www.icsbook.info/read-book/320
https://www.icsbook.info/read-book/324
https://www.icsbook.info/read-book/744
https://www.icsbook.info/read-book/721
https://www.icsbook.info/read-book/530
https://www.icsbook.info/read-book/321
https://www.icsbook.info/read-book/331
https://www.icsbook.info/read-book/338
https://www.icsbook.info/read-book/342
https://www.icsbook.info/read-book/291
https://www.icsbook.info/read-book/292
https://www.icsbook.info/read-book/293
https://www.icsbook.info/read-book/333
https://www.icsbook.info/read-book/334
https://www.icsbook.info/read-book/335
https://www.icsbook.info/read-book/337
https://www.icsbook.info/read-book/343
https://www.icsbook.info/read-book/329
https://www.icsbook.info/read-book/314
https://www.icsbook.info/read-book/315
https://www.icsbook.info/read-book/316
https://www.icsbook.info/read-book/317
https://www.icsbook.info/read-book/707
https://www.icsbook.info/read-book/351
https://www.icsbook.info/read-book/352
https://www.icsbook.info/read-book/350
https://www.icsbook.info/read-book/326
https://www.icsbook.info/read-book/327
https://www.icsbook.info/read-book/472
https://www.icsbook.info/read-book/332
https://www.icsbook.info/read-book/507
https://www.icsbook.info/read-book/508
https://www.icsbook.info/read-book/509
https://www.icsbook.info/read-book/510
https://www.icsbook.info/read-book/511
https://www.icsbook.info/read-book/512
https://www.icsbook.info/read-book/513
https://www.icsbook.info/read-book/514
https://www.icsbook.info/read-book/515
https://www.icsbook.info/read-book/363
https://www.icsbook.info/read-book/362
https://www.icsbook.info/read-book/645
https://www.icsbook.info/read-book/722
https://www.icsbook.info/read-book/723
https://www.icsbook.info/read-book/724
https://www.icsbook.info/read-book/361
https://www.icsbook.info/read-book/456
https://www.icsbook.info/read-book/356
https://www.icsbook.info/read-book/286
https://www.icsbook.info/read-book/699
https://www.icsbook.info/read-book/365
https://www.icsbook.info/read-book/28
https://www.icsbook.info/read-book/27
https://www.icsbook.info/read-book/366
https://www.icsbook.info/read-book/675
https://www.icsbook.info/read-book/367
https://www.icsbook.info/read-book/369
https://www.icsbook.info/read-book/700
https://www.icsbook.info/read-book/370
https://www.icsbook.info/read-book/368
https://www.icsbook.info/read-book/372
https://www.icsbook.info/read-book/32
https://www.icsbook.info/read-book/373
https://www.icsbook.info/read-book/322
https://www.icsbook.info/read-book/371
https://www.icsbook.info/read-book/741
https://www.icsbook.info/read-book/374
https://www.icsbook.info/read-book/375
https://www.icsbook.info/read-book/376
https://www.icsbook.info/read-book/380
https://www.icsbook.info/read-book/381
https://www.icsbook.info/read-book/382
https://www.icsbook.info/read-book/326
https://www.icsbook.info/read-book/644
https://www.icsbook.info/read-book/377
https://www.icsbook.info/read-book/726
https://www.icsbook.info/read-book/386
https://www.icsbook.info/read-book/725
https://www.icsbook.info/read-book/387
https://www.icsbook.info/read-book/385
https://www.icsbook.info/read-book/384
https://www.icsbook.info/read-book/701
https://www.icsbook.info/read-book/383
https://www.icsbook.info/read-book/389
https://www.icsbook.info/read-book/390
https://www.icsbook.info/read-book/642
https://www.icsbook.info/read-book/393
https://www.icsbook.info/read-book/439
https://www.icsbook.info/read-book/391
https://www.icsbook.info/read-book/742
https://www.icsbook.info/read-book/740
https://www.icsbook.info/read-book/388
https://www.icsbook.info/read-book/271
https://www.icsbook.info/read-book/396
https://www.icsbook.info/read-book/394
https://www.icsbook.info/read-book/395
https://www.icsbook.info/read-book/519
https://www.icsbook.info/read-book/253
https://www.icsbook.info/read-book/280
https://www.icsbook.info/read-book/398
https://www.icsbook.info/read-book/400
https://www.icsbook.info/read-book/281
https://www.icsbook.info/read-book/399
https://www.icsbook.info/read-book/727
https://www.icsbook.info/read-book/641
https://www.icsbook.info/read-book/733
https://www.icsbook.info/read-book/734
https://www.icsbook.info/read-book/735
https://www.icsbook.info/read-book/736
https://www.icsbook.info/read-book/584
https://www.icsbook.info/read-book/305
https://www.icsbook.info/read-book/302
https://www.icsbook.info/read-book/448
https://www.icsbook.info/read-book/739
https://www.icsbook.info/read-book/304
https://www.icsbook.info/read-book/306
https://www.icsbook.info/read-book/730
https://www.icsbook.info/read-book/310
https://www.icsbook.info/read-book/262
https://www.icsbook.info/read-book/747
https://www.icsbook.info/read-book/729
https://www.icsbook.info/read-book/737
https://www.icsbook.info/read-book/311
https://www.icsbook.info/read-book/312
https://www.icsbook.info/read-book/732
https://www.icsbook.info/read-book/301
https://www.icsbook.info/read-book/731
https://www.icsbook.info/read-book/687
https://www.icsbook.info/read-book/686
https://www.icsbook.info/read-book/278
https://www.icsbook.info/read-book/646
https://www.icsbook.info/read-book/279